In [5]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import math

In [8]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.
simulator = AerSimulator()

# Keep the exact same helper functions as the Plain notebook
def quantum_rng(num_bits):
    qc = QuantumCircuit(num_bits, num_bits)
    qc.h(range(num_bits))
    qc.measure(range(num_bits), range(num_bits))
    result = simulator.run(qc, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]
    return [int(b) for b in bitstring[::-1]]

def encode_qubits(bits, bases):
    circuits = []
    for i in range(len(bits)):
        qc = QuantumCircuit(1, 1)
        if bits[i] == 1: qc.x(0)
        if bases[i] == 1: qc.h(0)
        circuits.append(qc)
    return circuits

def measure_qubits(circuits, bases):
    measured_bits = []
    for i in range(len(circuits)):
        qc = circuits[i]
        if bases[i] == 1: qc.h(0)
        qc.measure(0, 0)
        result = simulator.run(qc, shots=1).result()
        measured_bits.append(int(list(result.get_counts().keys())[0]))
    return measured_bits

# ==========================================
# SIMULATION: BB84 WITH ATTACKER (EVE)
# ==========================================
num_qubits = 100 # Increased to get a stable error percentage

# --- ALICE ---
alice_bits = quantum_rng(num_qubits)
alice_bases = quantum_rng(num_qubits)
qubits_from_alice = encode_qubits(alice_bits, alice_bases)

# --- EVE (THE ATTACKER) ---
# Eve intercepts the qubits, guesses bases, measures, and resends
eve_bases = quantum_rng(num_qubits)
eve_bits = measure_qubits(qubits_from_alice, eve_bases)
qubits_from_eve = encode_qubits(eve_bits, eve_bases)

# --- BOB ---
# Bob receives the corrupted qubits from Eve, not knowing they were intercepted
bob_bases = quantum_rng(num_qubits)
bob_bits = measure_qubits(qubits_from_eve, bob_bases)

# --- PUBLIC DISCUSSION (SIFTING) ---
alice_key = []
bob_key = []

for i in range(num_qubits):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

# --- RESULTS & THRESHOLD CHECK ---
errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
error_rate = errors / len(alice_key) if len(alice_key) > 0 else 0

print("--- PROTOCOL RESULTS ---")
print(f"Sifted Key Length: {len(alice_key)}")
print(f"Total Errors:      {errors}")
print(f"Error Rate:        {error_rate * 100:.2f}%")

THRESHOLD = 0.15 # 15% threshold for reporting an attack
print("\n--- SECURITY CHECK ---")
if error_rate > THRESHOLD:
    print(f"ATTACK DETECTED! Error rate exceeds the {THRESHOLD*100}% threshold.")
    print("Alice and Bob abort the protocol and discard the key.")
else:
    print("Key appears secure. No attack detected.")

--- PROTOCOL RESULTS ---
Sifted Key Length: 45
Total Errors:      8
Error Rate:        17.78%

--- SECURITY CHECK ---
ATTACK DETECTED! Error rate exceeds the 15.0% threshold.
Alice and Bob abort the protocol and discard the key.
